In [1]:
# ======================================================================
# CLEAN PLANNING DATA: one ML-ready table, one row per planning application
# ======================================================================
# Input:  processed_dataset\planning_dataset\planning_2021_onwards.csv
# Output: clean_dataset\planning\planning_clean.csv
#
# What this table is for (from the proposal, Student 2):
#   - TF-IDF / n-grams + LDA topics      -> text: DevelopmentDescription
#   - ElasticNet regression              -> target: days_to_decision
#   - Linear SVM classification          -> target: is_refused (granted vs refused)
#
# Rules for this notebook:
#   - fix the data (spaces, labels, types, impossible values, duplicates)
#   - do NOT fill missing values, scale or encode here (done in the model Pipeline)
#   - leave out anything recorded AFTER the decision (grant date, appeals...),
#     because the proposal says those would leak the answer
#
# Needs 00_county_lookup.ipynb to be run first (uses county_lookup.csv)
# ======================================================================

import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)

base = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project\dataSet'

planning_file = base + r'\processed_dataset\planning_dataset\planning_2021_onwards.csv'
clean         = base + r'\clean_dataset'

In [2]:
planning = pd.read_csv(planning_file, low_memory=False)

print(planning.shape)
planning.head(3)

(225141, 36)


,OBJECTID,PlanningAuthority,ApplicationNumber,DevelopmentDescription,DevelopmentAddress,DevelopmentPostcode,ApplicationStatus,ApplicationType,Decision,LandUseCode,AreaofSite,NumResidentialUnits,OneOffHouse,FloorArea,ReceivedDate,WithdrawnDate,DecisionDate,DecisionDueDate,GrantDate,ExpiryDate,AppealRefNumber,AppealStatus,AppealDecision,AppealDecisionDate,AppealSubmittedDate,FIRequestDate,FIRecDate,LinkAppDetails,OneOffKPI,ETL_DATE,SiteId,word_count,county,year,quarter,days_to_decision
0,632,Carlow County Council,21393,Permission is sought for vehicular entrance in to zoned lands (to replace agricultural entrance / repositioning prev...,"Dublin Road , Pollerton Big , CarlowTown",NaN,APPLICATION FINALISED,PERMISSION,REFUSED,,NaN,0.0,,0.0,2021-10-15,NaN,2022-10-19,2022-10-20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-12-07,2022-08-26,http://www.eplanning.ie/CarlowCC/AppFileRefDetails/21393/0,No,2026-10-20,NaN,43,Carlow,2021,2021Q4,369.0
1,886,Carlow County Council,2445,"alterations and additions to an existing single storey dwelling house, namely erection of a single storey extension ...","Ballyveal , Ballon , Co. Carlow",NaN,APPLICATION FINALISED,PERMISSION,CONDITIONAL,,NaN,1.0,,10.0,2024-10-17,NaN,2024-12-09,2024-12-11,2025-01-20,2030-01-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.eplanning.ie/CarlowCC/AppFileRefDetails/2445/0,No,2026-10-20,NaN,49,Carlow,2024,2024Q4,53.0
2,887,Carlow County Council,2446,"construct a single-storey domestic dwelling, with on-site wastewater treatment system compliant with current EPA req...","Ballypierce , Kildavin , Co.Carlow",NaN,WITHDRAWN,PERMISSION,NaN,,NaN,0.0,,0.0,2024-10-29,2024-11-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.eplanning.ie/CarlowCC/AppFileRefDetails/2446/0,No,2026-10-20,NaN,31,Carlow,2024,2024Q4,NaN


In [3]:
# Removing extra spaces from all text. Different councils type the same
# value with spaces at the end ('GRANT PERMISSION   '), which made the
# same label look like 3 different ones.

text_cols = []
for col in planning.columns:
    if planning[col].dtype == 'object' or str(planning[col].dtype) == 'str':
        text_cols.append(col)

for col in text_cols:
    planning[col] = planning[col].str.strip()
    planning[col] = planning[col].replace('', np.nan)

print('Text columns cleaned:', len(text_cols))

Text columns cleaned: 28


In [4]:
# Duplicates: some applications appear twice with everything the same except
# OBJECTID (the ArcGIS row number). Those are the same application twice.

cols_without_id = []
for col in planning.columns:
    if col != 'OBJECTID':
        cols_without_id.append(col)

print('Exact duplicates (ignoring OBJECTID):', planning.duplicated(subset=cols_without_id).sum())

rows_before = len(planning)
planning = planning.drop_duplicates(subset=cols_without_id)
print('Removed', rows_before - len(planning), 'rows. Rows left:', len(planning))

Exact duplicates (ignoring OBJECTID): 190
Removed 190 rows. Rows left: 224951


In [5]:
# Dates come back from CSV as text, so converting again.

for col in ['ReceivedDate', 'DecisionDate']:
    planning[col] = pd.to_datetime(planning[col], errors='coerce')

print('Received from', planning['ReceivedDate'].min(), 'to', planning['ReceivedDate'].max())
print('Latest decision date:', planning['DecisionDate'].max())

Received from 2021-01-04 00:00:00 to 2026-10-19 00:00:00
Latest decision date: 2075-12-03 00:00:00


In [6]:
# County: using the shared county_lookup.csv so the county names and ids are
# exactly the same as in BER, PPR and rent.

lookup = pd.read_csv(clean + r'\county_lookup.csv')
lookup = lookup[lookup['source'] == 'planning'][['raw_name', 'county', 'county_id']]

planning = planning.drop(columns=['county'])   # replaced by the lookup version
planning = planning.merge(lookup, left_on='PlanningAuthority', right_on='raw_name', how='left')
planning = planning.drop(columns=['raw_name'])

print('Rows without a county:', planning['county'].isnull().sum())   # should be 0
print(planning['county'].nunique(), 'counties')

Rows without a county: 0
26 counties


In [7]:
# The Decision column is very messy: about 180 different spellings for a
# handful of real outcomes, e.g.
#   'CONDITIONAL', 'GRANT PERMISSION', 'Granted (Conditional)', 'Grant with Conditions'
# Looking at the most common ones first.

planning['Decision'].value_counts(dropna=False).head(40)

Decision
CONDITIONAL                               108840
NaN                                        46909
GRANT PERMISSION                           19677
REFUSED                                    13599
Granted (Conditional)                       6392
APPLICATION DECLARED INVA                   4328
Grant Permission                            3780
REFUSE PERMISSION                           2820
Grant with Conditions                       1871
Refused                                     1853
Conditional Permission                      1777
APPLICATION DECLARED INVALID                1414
UNCONDITIONAL                               1187
GRANT PERMISSION FOR RETENTION              1017
Invalid Application                          930
DECLARE APPLICATION INVALID                  799
GRANT RETENTION PERMISSIO                    742
GRANT PERMISSION & GRANT RETENTION           529
Refuse Permission                            363
Grant Permission for Retention               341
GRANT PERMI

In [8]:
# Grouping all decision labels into a few clear outcomes by looking for key
# words. The order matters (e.g. 'App withdrawn as no AI recd' must be
# Withdrawn, and 'GRANT CERTIFICATE OF EXEMPTION' is not a normal grant).
#
#   Pending    -> no decision yet, or still waiting for further information
#   Invalid    -> application declared invalid
#   Withdrawn  -> applicant withdrew / deemed withdrawn
#   Exempt     -> Section 5 / exemption decisions (not a normal planning decision)
#   Split      -> part granted and part refused
#   Refused
#   Granted    -> conditional or unconditional permission / approval
#   Other      -> anything else (quashed by court, decided by another body...)

def decision_group(text):
    if pd.isnull(text):
        return 'Pending'

    t = text.upper()

    if 'INVA' in t:                     # some labels are cut short: 'APPLICATION DECLARED INVA'
        return 'Invalid'
    if 'WITHDRAW' in t:
        return 'Withdrawn'
    if 'EXEMPT' in t or t.startswith('S5') or 'SECTION 5' in t or 'S179A' in t:
        return 'Exempt'
    if 'SPLIT' in t or ('GRANT' in t and ('REFUSE' in t or 'REF.' in t or 'REF ' in t)):
        return 'Split'
    if ('INFORMATION' in t or 'REQ AI' in t or 'REQUEST AI' in t or 'CLARIFICATION' in t
            or 'EXTENSION OF TIME' in t or 'TIME EXT' in t):
        return 'Pending'
    if 'REFUSE' in t or 'REJECTED' in t:
        return 'Refused'
    if 'GRANT' in t or 'CONDITIONAL' in t or 'APPROV' in t:
        return 'Granted'
    return 'Other'


planning['decision_group'] = planning['Decision'].apply(decision_group)

print(planning['decision_group'].value_counts())

decision_group
Granted      147445
Pending       47295
Refused       19512
Invalid        8299
Exempt          994
Withdrawn       929
Split           447
Other            30
Name: count, dtype: int64


In [9]:
# Checking the grouping: every raw label next to the group it went into.
# Worth scrolling through to make sure nothing landed in the wrong place.

check = planning.groupby(['decision_group', 'Decision']).size().reset_index(name='count')
check = check.sort_values(['decision_group', 'count'], ascending=[True, False])
check

,decision_group,Decision,count
1,Exempt,Declared Exempt,254
11,Exempt,S5 DEC EXEMPT,239
4,Exempt,Declared Not Exempt,142
8,Exempt,GRANT CERTIFICATE OF EXEMPTION,134
12,Exempt,S5 DEC NOT EXEMPT,101
7,Exempt,GRANT CERT OF EXEMPTION (<=4 unit),33
14,Exempt,S5 SPLIT DECISION,32
13,Exempt,S5 REQ AI,20
2,Exempt,Declared Exempt & Declared Not Exempt,17
6,Exempt,GRANT CERT OF EXEMPTION (0.1 Hec ),12


In [10]:
# Classification target for the Linear SVM: refused (1) vs granted (0).
# Only applications with a clear Granted or Refused outcome get a label.
# Everything else (pending, invalid, withdrawn, split, exempt) stays empty,
# so it can still be used for the text mining but not for this model.

planning['is_refused'] = np.nan
planning.loc[planning['decision_group'] == 'Granted', 'is_refused'] = 0
planning.loc[planning['decision_group'] == 'Refused', 'is_refused'] = 1

print(planning['is_refused'].value_counts(dropna=False))
print()
print('Refusal rate among decided:', round(planning['is_refused'].mean(), 3))

is_refused
0.0    147445
NaN     57994
1.0     19512
Name: count, dtype: int64

Refusal rate among decided: 0.117


In [11]:
# Regression target for ElasticNet: days from received date to decision date.
# Problems found:
#   - some decision dates are BEFORE the received date (negative days)
#   - one decision date is in 2075
# Those can't be real, so they become missing.
# Also only keeping days for real decisions (Granted / Refused / Split); an
# 'invalid' or 'withdrawn' date is not a decision on the application.

planning['days_to_decision'] = (planning['DecisionDate'] - planning['ReceivedDate']).dt.days

print('Negative days:', (planning['days_to_decision'] < 0).sum())
print('Decision dates after Oct 2026:', (planning['DecisionDate'] > '2026-10-31').sum())

planning.loc[planning['days_to_decision'] < 0, 'days_to_decision'] = np.nan
planning.loc[planning['DecisionDate'] > '2026-10-31', 'days_to_decision'] = np.nan
planning.loc[~planning['decision_group'].isin(['Granted', 'Refused', 'Split']), 'days_to_decision'] = np.nan

print(planning['days_to_decision'].describe().round(1))

Negative days: 45
Decision dates after Oct 2026: 1
count    167158.0
mean         91.9
std          71.1
min           0.0
25%          51.0
50%          55.0
75%         111.0
max        1977.0
Name: days_to_decision, dtype: float64


In [12]:
# Application type: same problem as Decision (PERMISSION, Permission,
# 'Permission   '...). Grouping into a few clear types.

def application_group(text):
    if pd.isnull(text):
        return np.nan

    t = text.upper()

    if 'CONSEQ' in t or 'ON FOOT' in t or 'FOLLOWING GRANT' in t:
        return 'Consequent on outline'
    if 'EXTENSION' in t:
        return 'Extension of duration'
    if 'EXEMPT' in t or 'SECTION 5' in t or 'SECT. 5' in t or '179' in t:
        return 'Exemption'
    if 'SHD' in t or 'LRD' in t or 'SDZ' in t or 'STRATEGIC' in t:
        return 'Large scale (SHD/LRD/SDZ)'
    if 'PART 8' in t or 'PART VIII' in t or 'PART VLLL' in t or 'PART X' in t or 'PART 10' in t:
        return 'Council own development'
    if 'OUTLINE' in t:
        return 'Outline permission'
    if 'RETENTION' in t and 'PERMISSION AND' in t:
        return 'Permission and retention'
    if 'RETENTION' in t:
        return 'Retention'
    if 'PERMISSION' in t:
        return 'Permission'
    return 'Other'


planning['application_group'] = planning['ApplicationType'].apply(application_group)

print(planning['application_group'].value_counts(dropna=False))

application_group
Permission                   178360
Retention                     37167
Extension of duration          3593
Outline permission             2187
Exemption                      1088
Permission and retention        851
Consequent on outline           845
Large scale (SHD/LRD/SDZ)       523
NaN                             194
Council own development         106
Other                            37
Name: count, dtype: int64


In [13]:
# One-off house: written as Y, Yes, One, Single House, No... -> 1 / 0.
# Blank stays missing.

print(planning['OneOffHouse'].value_counts(dropna=False))

one_off_map = {'Y': 1, 'Yes': 1, 'One': 1, 'Single House': 1, 'No': 0, 'N': 0}
planning['one_off_house'] = planning['OneOffHouse'].map(one_off_map)

print(planning['one_off_house'].value_counts(dropna=False))

OneOffHouse
NaN             203385
Y                12467
No                6022
One               1769
Single House       994
Yes                314
Name: count, dtype: int64
one_off_house
NaN    203385
1.0     15544
0.0      6022
Name: count, dtype: int64


In [14]:
# Text column: keeping the original description, and adding a lower-case
# version with extra spaces removed (easier for TF-IDF / LDA later).
# Stop words, n-grams etc. are left for the text-mining notebook.

planning['clean_text'] = planning['DevelopmentDescription'].fillna('').str.lower()
planning['clean_text'] = planning['clean_text'].str.split().str.join(' ')

planning['word_count'] = planning['clean_text'].str.split().str.len()

print('Average words:', round(planning['word_count'].mean(), 1))
print('Median words: ', planning['word_count'].median())
print('Empty descriptions:', (planning['word_count'] == 0).sum())

Average words: 41.3
Median words:  27.0
Empty descriptions: 1


In [15]:
# Residential flag (the proposal mentions residential-project filters).
# An application counts as residential if it has residential units, is a
# one-off house, or the description mentions a home.

home_words = ['dwelling', 'house', 'apartment', 'residential', 'housing', 'home']

mentions_home = planning['clean_text'].str.contains('|'.join(home_words))

planning['is_residential'] = 0
planning.loc[planning['NumResidentialUnits'] > 0, 'is_residential'] = 1
planning.loc[planning['one_off_house'] == 1, 'is_residential'] = 1
planning.loc[mentions_home, 'is_residential'] = 1

print(planning['is_residential'].value_counts(normalize=True).round(3))

is_residential
1    0.646
0    0.354
Name: proportion, dtype: float64


In [16]:
# Quarter and year from the received date (same format as BER, PPR, rent).

planning['year'] = planning['ReceivedDate'].dt.year
planning['quarter'] = planning['ReceivedDate'].dt.to_period('Q').astype(str)

planning['year'].value_counts().sort_index()

year
2021    44333
2022    37428
2023    36189
2024    38001
2025    39685
2026    29315
Name: count, dtype: int64

In [17]:
# Numeric columns: floor area and site area have some huge values
# (site area up to 10 billion). Not deleting them here, just looking.
# Outliers are handled at the modelling stage.

planning[['NumResidentialUnits', 'FloorArea', 'AreaofSite']].describe().round(1)

,NumResidentialUnits,FloorArea,AreaofSite
count,165579.0,165153.0,1.746410e+05
mean,1.2,577.0,7.768090e+04
std,12.0,83334.0,2.659248e+07
min,0.0,0.0,0.000000e+00
25%,0.0,0.0,2.000000e-01
50%,0.0,0.0,4.000000e-01
75%,1.0,123.0,7.900000e+00
max,1126.0,33322425.0,1.096980e+10


In [18]:
# Choosing the columns for the clean table.
# Left out on purpose:
#   GrantDate, ExpiryDate, all Appeal columns, WithdrawnDate -> after the decision (leakage)
#   ApplicationStatus      -> describes where the case ended up (leakage)
#   FIRequestDate, FIRecDate -> only known while the case is being processed
#   DevelopmentAddress, DevelopmentPostcode -> address details, not needed (data minimisation)
#   LandUseCode, OneOffKPI, LinkAppDetails, SiteId, ETL_DATE, DecisionDueDate -> mostly empty or admin
#   Decision, ApplicationType, OneOffHouse -> replaced by the cleaned versions

keep = ['OBJECTID', 'PlanningAuthority', 'ApplicationNumber', 'county', 'county_id',
        'ReceivedDate', 'DecisionDate', 'year', 'quarter',
        'application_group', 'one_off_house', 'NumResidentialUnits', 'FloorArea', 'AreaofSite',
        'is_residential',
        'DevelopmentDescription', 'clean_text', 'word_count',
        'decision_group', 'is_refused', 'days_to_decision']

planning_clean = planning[keep].rename(columns={'OBJECTID': 'application_id'})

print(planning_clean.shape)
planning_clean.head()

(224951, 21)


,application_id,PlanningAuthority,ApplicationNumber,county,county_id,ReceivedDate,DecisionDate,year,quarter,application_group,one_off_house,NumResidentialUnits,FloorArea,AreaofSite,is_residential,DevelopmentDescription,clean_text,word_count,decision_group,is_refused,days_to_decision
0,632,Carlow County Council,21393,Carlow,1,2021-10-15,2022-10-19,2021,2021Q4,Permission,NaN,0.0,0.0,NaN,0,Permission is sought for vehicular entrance in to zoned lands (to replace agricultural entrance / repositioning prev...,permission is sought for vehicular entrance in to zoned lands (to replace agricultural entrance / repositioning prev...,43,Refused,1.0,369.0
1,886,Carlow County Council,2445,Carlow,1,2024-10-17,2024-12-09,2024,2024Q4,Permission,NaN,1.0,10.0,NaN,1,"alterations and additions to an existing single storey dwelling house, namely erection of a single storey extension ...","alterations and additions to an existing single storey dwelling house, namely erection of a single storey extension ...",49,Granted,0.0,53.0
2,887,Carlow County Council,2446,Carlow,1,2024-10-29,NaT,2024,2024Q4,Permission,NaN,0.0,0.0,NaN,1,"construct a single-storey domestic dwelling, with on-site wastewater treatment system compliant with current EPA req...","construct a single-storey domestic dwelling, with on-site wastewater treatment system compliant with current epa req...",31,Pending,NaN,NaN
3,888,Carlow County Council,2447,Carlow,1,2024-10-31,NaT,2024,2024Q4,Permission,NaN,0.0,0.0,NaN,1,"the construction of a one and a half storey extension to side of existing dwelling, attic conversion and to include ...","the construction of a one and a half storey extension to side of existing dwelling, attic conversion and to include ...",33,Pending,NaN,NaN
4,889,Carlow County Council,2448,Carlow,1,2024-11-13,NaT,2024,2024Q4,Permission,NaN,0.0,0.0,NaN,1,the erection of two houses and two garages,the erection of two houses and two garages,8,Pending,NaN,NaN


In [19]:
# Missing values left (% per column). Not filling them here.

missing = (planning_clean.isnull().mean() * 100).round(1)
missing[missing > 0].sort_values(ascending=False)

one_off_house          90.4
FloorArea              26.6
NumResidentialUnits    26.4
is_refused             25.8
days_to_decision       25.7
AreaofSite             22.4
DecisionDate           20.4
application_group       0.1
dtype: float64

In [20]:
planning_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 224951 entries, 0 to 224950
Data columns (total 21 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   application_id          224951 non-null  int64         
 1   PlanningAuthority       224951 non-null  str           
 2   ApplicationNumber       224951 non-null  str           
 3   county                  224951 non-null  str           
 4   county_id               224951 non-null  int64         
 5   ReceivedDate            224951 non-null  datetime64[us]
 6   DecisionDate            179144 non-null  datetime64[us]
 7   year                    224951 non-null  int32         
 8   quarter                 224951 non-null  str           
 9   application_group       224757 non-null  str           
 10  one_off_house           21566 non-null   float64       
 11  NumResidentialUnits     165579 non-null  float64       
 12  FloorArea               165153 non-null  

In [21]:
# Saving the clean planning table.

save_folder = clean + r'\planning'
os.makedirs(save_folder, exist_ok=True)

planning_clean.to_csv(save_folder + r'\planning_clean.csv', index=False)
print('Saved', planning_clean.shape, 'to', save_folder + r'\planning_clean.csv')

Saved (224951, 21) to C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project\dataSet\clean_dataset\planning\planning_clean.csv
